In [1]:
import sentence_transformers
import array
import oracledb
from sentence_transformers import SparseEncoder

def tensor_to_oracle_sparse_vector(inputTensor):
     """Convert torch tensor to oracledb.SparseVector"""
     # Convert torch tensor to oracledb.SparseVector by isolating indices and values
     # Coalesce the tensor first to merge duplicate indices
     t = inputTensor.coalesce()
     num_dimensions = t.size(1) # Assuming 2D, using the second dimension for vector size
     indices = t.indices()[1].tolist() # Get column indices
     values = t.values().tolist()
     
     return oracledb.SparseVector(
          num_dimensions,
          indices,
          array.array('f', values) 
     )

def encode_sentence_to_tensor(sentence_text, model):
     """Encode a sentence to sparse tensor representation"""
     sentence = [sentence_text]
     # Encode the sentence to get sparse representation as a pytorch tensor
     t = model.encode(sentence)
     print(f"print the Pytorch tensor:\n {t}")
     return t

def encode_sentence_to_vector(sentence_text, model):
     """Encode a sentence to oracledb.SparseVector representation"""
     t = encode_sentence_to_tensor(sentence_text, model)
     v = tensor_to_oracle_sparse_vector(t)
     return v

def print_sparse_vector_info(sparse_vector):
     """Print information about an oracledb.SparseVector"""
     print("Oracle Sparse Vector Details:")
     print(f"  Dimensions: {sparse_vector.num_dimensions}")
     print(f"  Indices: {sparse_vector.indices}")
     print(f"  Values: {sparse_vector.values}")
     print(f"  Print the oracle sparse vector:\n {sparse_vector}")

# Initialize model
# citation: Damodaran, P. (2024). Splade_PP_en_v2: Independent Implementation of SPLADE++ Model (`a.k.a splade-cocondenser* and family`) for the Industry setting. (Version 2.0.0) [Computer software].
model = SparseEncoder("prithivida/Splade_PP_en_v2")

s = "The sky is blue like an orange."
print(f"Encoding sentence: {s}")
v = encode_sentence_to_vector(s, model)
print_sparse_vector_info(v)

s = "Many species of sea turtles are currently endangered."
print(f"\nEncoding sentence: {s}")
v = encode_sentence_to_vector(s, model)
print_sparse_vector_info(v)



Encoding sentence: The sky is blue like an orange.
print the Pytorch tensor:
 tensor(indices=tensor([[    0,     0,     0,     0,     0,     0,     0,     0,
                            0,     0,     0,     0,     0,     0,     0,     0,
                            0,     0,     0,     0,     0,     0,     0,     0,
                            0,     0,     0,     0,     0,     0,     0,     0,
                            0,     0,     0,     0,     0,     0,     0],
                       [ 1012,  1996,  2066,  2305,  2317,  2417,  2422,  2621,
                         2630,  2686,  2806,  3103,  3311,  3609,  3712,  3746,
                         3959,  4169,  4281,  4323,  4432,  4589,  4633,  4774,
                         5132,  5304,  5443,  6087,  6112,  6454,  6459,  7224,
                         8044,  8703, 10098, 12799, 15717, 20418, 20639]]),
       values=tensor([0.1441, 0.3146, 1.3771, 0.1830, 0.1249, 0.2759, 0.0088,
                      0.4136, 2.4873, 0.1360, 0.0387, 

In [5]:
mySentence = "The sky was clear after the rain."
myTensor = model.encode(mySentence)
print(myTensor.shape)
# [1, 30522]
print(myTensor)

decoded_sentence = model.decode(myTensor)

print(f"Number of actual dimensions: {len(decoded_sentence)}")
decoded_sentence_rounded = [(token, round(score, 2)) for token, score in decoded_sentence]
print("SPLADE Bag Of Words details:\n", decoded_sentence_rounded)

torch.Size([30522])
tensor(indices=tensor([[ 1012,  1996,  2001,  2008,  2020,  2025,  2044,  2145,
                         2156,  2300,  2305,  2422,  2621,  2851,  3154,  3609,
                         3712,  4040,  4432,  4542,  4550,  4633,  4658,  4860,
                         4954,  5475,  6087,  6112,  8044,  9666, 10101, 11094,
                        15717, 15811]]),
       values=tensor([0.3288, 0.1541, 0.4763, 0.0081, 0.2338, 0.2021, 2.2545,
                      0.2781, 0.0339, 0.0850, 0.0494, 0.2413, 0.2154, 0.2738,
                      2.6487, 0.4018, 2.1772, 0.0722, 0.1121, 2.5850, 0.9133,
                      0.9923, 0.1443, 0.2947, 0.3360, 0.1122, 0.0907, 0.4053,
                      0.4942, 0.4368, 0.9526, 0.1619, 1.2611, 1.3383]),
       size=(30522,), nnz=34, layout=torch.sparse_coo)
Number of actual dimensions: 34
SPLADE Bag Of Words details:
 [('clear', 2.65), ('rain', 2.58), ('after', 2.25), ('sky', 2.18), ('rains', 1.34), ('skies', 1.26), ('weather', 0.99),